# Fitting psychometric models on real data

Multiple fitting scenarios with `piepy.fitting`. Everything routes through one entry point:

```python
fit(model, x, y, *, n=None, method="auto", ci="auto", confidence=0.95) -> FitResult
```

- **Models** (`get_model` / by name): `"logistic"` (4p: `x0,k,lapse_low,lapse_high`), `"weibull"`
  (4p on `|x|`: `alpha,beta,guess,lapse` — the detection shape), `"erf"` (3p: `mu,sigma,lapse` —
  the discrimination shape).
- **method**: `"auto"` = binomial **MLE** when per-level trial counts `n=` are given, else
  least-squares (**LSQ**).
- **ci**: `"auto"` = covariance for LSQ, bootstrap for MLE. Also `"covariance"`, `"bootstrap"`,
  `"none"`.
- **FitResult**: `.params`, `.param_ci`, `.gof` (`r2`, `n_points`, + `log_likelihood` for MLE),
  `.predict(x)`, `.curve(n)`.

**Real data:** point `~/.piepy/config.json` `paths.presentation` at your dirs. Each scenario tries
a real session first and falls back to a simulated one (with a printed note) so the notebook always
runs end-to-end.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from piepy.core.registry import get_session_class
from piepy.simulations.session import simulate_session
from piepy.stats import aggregate, Median
from piepy.fitting import fit, get_model, MODELS

print("models:", list(MODELS))

In [ ]:
exp_name = "240627_KC148_detect_opto120_HVA__1P_KC"
sess = get_session_class("detection")(exp_name, load_flag=True)
df = sess.concatenate_runs("detection")
print(exp_name, "->", df.shape)

In [ ]:
df.head()

In [ ]:
aggregate(df, group=["animalid","stim_type","opto_pattern","signed_contrast","stim_side"], metrics=[Median], rate="outcome",success="hit")

In [ ]:
def psy(df: pl.DataFrame, x: str, rate: str, success=None) -> pl.DataFrame:
    """P(success) per stimulus level with Wilson CIs (drops null x)."""
    return aggregate(df.filter(pl.col(x).is_not_null()), group=x, rate=rate, success=success)


def plot_fit(ax, agg, xcol, fr=None, label="", marker="o"):
    x = agg[xcol].to_numpy()
    y = agg["value"].to_numpy()
    yerr = np.vstack([y - agg["ci_low"].to_numpy(), agg["ci_high"].to_numpy() - y])
    ax.errorbar(x, y, yerr=yerr, fmt=marker, capsize=3, label=f"{label} data".strip())
    if fr is not None:
        xx, yy = fr.curve(200)
        ax.plot(xx, yy, "-", label=f"{label} fit".strip())
    ax.set_ylim(-0.02, 1.02)

In [ ]:
# A detection session (P(hit) vs contrast). opto_ratio>0 so the opto-split scenario has data.
det = load_session(
    "detection",
    "240627_KC148_detect_opto120_HVA__1P_KC",
    n_trials=1200, opto_ratio=0.3, early_rate=0.1, seed=0,
)
stim = det.filter(pl.col("outcome") != "early")  # drop no-stimulus (early) trials
print("outcomes:", det["outcome"].value_counts().sort("outcome").to_dict(as_series=False))

## Scenario 1 — Logistic: LSQ vs binomial MLE

Same data, two estimators. Pass `n=` (per-level trial counts) to switch from least-squares to
binomial maximum-likelihood — MLE weights levels by how many trials they have, so sparse extremes
stop dominating the curve.

In [ ]:
a = psy(stim, "signed_contrast", "outcome", success="hit")
x, y, n = a["signed_contrast"].to_numpy(), a["value"].to_numpy(), a["n"].to_numpy()

f_lsq = fit("logistic", x, y)            # no n -> least squares
f_mle = fit("logistic", x, y, n=n)       # n given -> binomial MLE

for name, fr in [("LSQ", f_lsq), ("MLE", f_mle)]:
    p = fr.params
    print(f"{name}: x0={p['x0']:+.3f} k={p['k']:.2f} "
          f"lapse=({p['lapse_low']:.3f},{p['lapse_high']:.3f}) r2={fr.gof['r2']:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
plot_fit(ax, a, "signed_contrast", f_lsq, "LSQ")
plot_fit(ax, a, "signed_contrast", f_mle, "MLE")
ax.axhline(0.5, ls=":", c="gray"); ax.axvline(0, ls=":", c="gray")
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(hit)"); ax.legend(); fig.tight_layout()

## Scenario 2 — Confidence intervals: covariance vs bootstrap

`ci="covariance"` is only valid for LSQ (from the fit covariance). MLE fits get a **bootstrap** CI
(resample binomial counts, refit `n_boot` times). Compare interval widths on the slope `k`.

In [ ]:
f_cov  = fit("logistic", x, y, method="lsq", ci="covariance")
f_boot = fit("logistic", x, y, n=n, method="mle", ci="bootstrap", n_boot=400, seed=0)

for name, fr in [("covariance (LSQ)", f_cov), ("bootstrap (MLE)", f_boot)]:
    lo, hi = fr.param_ci["k"]
    print(f"{name:18s} k={fr.params['k']:.2f}  95% CI=({lo:.2f}, {hi:.2f})  width={hi-lo:.2f}")

## Scenario 3 — Weibull on hit-rate vs |contrast|

The detection shape: collapse left/right and fit P(hit) against **unsigned** contrast. `alpha` is the
threshold (contrast at the curve's rise), `beta` the steepness.

In [ ]:
aw = psy(stim, "contrast", "outcome", success="hit")  # 'contrast' is the unsigned magnitude
xw, yw, nw = aw["contrast"].to_numpy(), aw["value"].to_numpy(), aw["n"].to_numpy()
fw = fit("weibull", xw, yw, n=nw)
p = fw.params
print(f"weibull: alpha(threshold)={p['alpha']:.3f} beta(slope)={p['beta']:.2f} "
      f"guess={p['guess']:.3f} lapse={p['lapse']:.3f} r2={fw.gof['r2']:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
plot_fit(ax, aw, "contrast", fw, "weibull")
ax.axvline(p["alpha"], ls="--", c="r", label="alpha (threshold)")
ax.set_xlabel("|contrast|"); ax.set_ylabel("P(hit)"); ax.legend(); fig.tight_layout()

## Scenario 4 — Model comparison

Fit logistic / weibull / erf to the **same** signed-contrast data and rank by goodness-of-fit.
`r2` is comparable across all; `log_likelihood` (MLE only) rewards calibrated probabilities.

In [ ]:
rows, fits = [], {}
for name in ["logistic", "weibull", "erf"]:
    fr = fit(name, x, y, n=n)
    fits[name] = fr
    rows.append({"model": name, "n_params": get_model(name).n_params,
                 "r2": round(fr.gof["r2"], 4),
                 "log_lik": round(fr.gof.get("log_likelihood", float("nan")), 2)})
ranking = pl.DataFrame(rows).sort("r2", descending=True)
print(ranking)
best = ranking["model"][0]; print("best by r2:", best)

fig, ax = plt.subplots(figsize=(6, 4))
plot_fit(ax, a, "signed_contrast", None, "")
for name, fr in fits.items():
    xx, yy = fr.curve(200); ax.plot(xx, yy, label=name)
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(hit)"); ax.legend(); fig.tight_layout()

## Scenario 5 — Split fits: opto vs non-opto

The real experimental question — does perturbation shift threshold (`x0`) or change slope (`k`)?
Fit each condition separately and overlay.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for opto, lab, mk in [(0, "control", "o"), (1, "opto", "s")]:
    sub = stim.filter(pl.col("opto") == opto)
    if sub.height == 0:
        print(f"no {lab} trials"); continue
    ag = psy(sub, "signed_contrast", "outcome", success="hit")
    fr = fit("logistic", ag["signed_contrast"].to_numpy(), ag["value"].to_numpy(),
             n=ag["n"].to_numpy())
    print(f"{lab:8s}: x0={fr.params['x0']:+.3f}  k={fr.params['k']:.2f}")
    plot_fit(ax, ag, "signed_contrast", fr, lab, marker=mk)
ax.axhline(0.5, ls=":", c="gray"); ax.axvline(0, ls=":", c="gray")
ax.set_xlabel("signed contrast"); ax.set_ylabel("P(hit)"); ax.legend(); fig.tight_layout()

## Scenario 6 — Parameter recovery (validate the fitter)

Treat the fitted params as ground truth, simulate new outcomes at the **real** stimulus grid and
trial counts (`Model.sample`), refit, and check we get the truth back. Tight scatter on the
identity line = the fitter is unbiased for this design.

In [ ]:
model = get_model("logistic")
truth = np.array([f_mle.params[k_] for k_ in model.param_names])
rng = np.random.default_rng(0)
rec = []
for _ in range(40):
    # one Bernoulli draw per trial at each level, then collapse to per-level rates
    y_sim = np.array([model.sample(np.full(int(ni), xi), truth, rng=rng).mean()
                      for xi, ni in zip(x, n)])
    rec.append([fit("logistic", x, y_sim, n=n).params[k_] for k_ in model.param_names])
rec = np.array(rec)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax_, i, nm in zip(axes, [0, 1], ["x0", "k"]):
    ax_.axhline(truth[i], ls="--", c="r", label="truth")
    ax_.scatter(np.arange(len(rec)), rec[:, i], s=12)
    ax_.set_title(f"recovered {nm} (mean={rec[:, i].mean():.3f})"); ax_.legend()
fig.tight_layout()

## Scenario 7 — Discrimination: erf for P(right) vs signed feature

Discrimination uses the cumulative-Gaussian (`erf`): `mu` is the bias (point of subjective
equality), `sigma` the inverse sensitivity. We fit `P(right_choice)` against the signed
discriminated-feature difference (`diff_*`).

In [ ]:
disc = load_session("discrimination", "250318_VB101_1stim_opto120_ALRL__no_cam_VO",
                    n_trials=1000, seed=1)
diff_cols = [c for c in disc.columns if c.startswith("diff_")]
if diff_cols and "right_choice" in disc.columns:
    xcol = diff_cols[0]
    ad = psy(disc, xcol, "right_choice")          # right_choice already 0/1
    fe = fit("erf", ad[xcol].to_numpy(), ad["value"].to_numpy(), n=ad["n"].to_numpy())
    print(f"erf: mu(bias)={fe.params['mu']:+.3f}  sigma={fe.params['sigma']:.3f}  "
          f"lapse={fe.params['lapse']:.3f}  r2={fe.gof['r2']:.3f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_fit(ax, ad, xcol, fe, "erf")
    ax.axhline(0.5, ls=":", c="gray"); ax.axvline(fe.params["mu"], ls="--", c="r", label="mu (bias)")
    ax.set_xlabel(xcol); ax.set_ylabel("P(right)"); ax.legend(); fig.tight_layout()
else:
    print("no discrimination diff_*/right_choice columns available; skipping erf scenario")

## Cheat-sheet

| want | call |
|---|---|
| quick curve, few points | `fit(m, x, y)` (LSQ + covariance CI) |
| proper binomial fit | `fit(m, x, y, n=counts)` (MLE + bootstrap CI) |
| no CI (speed) | `fit(..., ci="none")` |
| reproducible bootstrap | `fit(..., ci="bootstrap", seed=0, n_boot=N)` |
| plotting curve | `xx, yy = fr.curve(200)` |
| predicted p at x | `fr.predict(x)` |
| compare models | rank `fr.gof["r2"]` (all) / `log_likelihood` (MLE) |

Pitfalls: `ci="covariance"` is LSQ-only; needs ≥ `n_params` distinct x-levels; `x` for `weibull` is
taken as `|x|`; drop `early` (no-stimulus) trials before fitting hit rate.